# Pointwise Modulo: truncated (`mod`) vs floored (`floor_mod`)

cuDNN's pointwise API provides **two flavors of modulo (remainder)**. They produce the same result
whenever the dividend `a` and the divisor `b` have the **same sign**, and differ **only** when `a`
and `b` have **opposite signs**.

## `graph.mod(a, b)` — truncated remainder

Truncated modulo, matching C's `fmod` / `%` and **`torch.fmod(a, b)`**. The quotient is rounded
toward zero, so the result takes the **sign of the dividend `a`** (backend enum
`CUDNN_POINTWISE_MOD`).

## `graph.floor_mod(a, b)` — floored remainder

Floored modulo, defined as

$$y = a - \lfloor a / b \rfloor \cdot b$$

matching Python's `%` and **`torch.remainder(a, b)`**. The quotient is rounded toward negative
infinity, so the result takes the **sign of the divisor `b`** (backend enum
`CUDNN_POINTWISE_FLOOR_MOD`).

## Worked example

For `a = [-7, 7, -7, 7]` and `b = [3, 3, -3, -3]`:

| `a` | `b` | `mod` = `torch.fmod(a, b)` (sign of `a`) | `floor_mod` = `torch.remainder(a, b)` (sign of `b`) |
|----:|----:|----------------------------------------:|----------------------------------------------------:|
| -7  |  3  |                                      -1 |                                                   2 |
|  7  |  3  |                                       1 |                                                   1 |
| -7  | -3  |                                      -1 |                                                  -1 |
|  7  | -3  |                                       1 |                                                  -2 |

The first and last rows have opposite-sign operands, so `mod` and `floor_mod` disagree there; the
middle two rows share a sign, so they agree.

## Prerequisites and Setup

This notebook requires an NVIDIA GPU and a cuDNN build whose backend provides the floored-modulo
pointwise mode `CUDNN_POINTWISE_FLOOR_MOD` (**cuDNN 9.26+**). On older backends the `floor_mod`
cell fails at build time; the truncated `mod` cell works on any backend that supports
`CUDNN_POINTWISE_MOD`.

If the backend does not support `floor_mod`, the `floor_mod` cells below catch `cudnnGraphNotSupportedError`
and are skipped gracefully instead of crashing the notebook.

**Environment setup:**

- **Option A — pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B — set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

In [ ]:
# get_ipython().system('pip install nvidia-cudnn-cu13')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system('pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu130')

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Failed to initialize NumPy")

import torch
import cudnn

# floor_mod requires a cuDNN backend that provides CUDNN_POINTWISE_FLOOR_MOD (cuDNN 9.26+).
print("cuDNN backend version:", cudnn.backend_version())

In [ ]:
# Illustrative operands, chosen so mod and floor_mod disagree on the opposite-sign entries.
# a and b share a sign in columns 1 and 2 (agree), and have opposite signs in columns 0 and 3 (differ).
device = "cuda"

a_gpu = torch.tensor([[-7.0, 7.0, -7.0, 7.0]], dtype=torch.float32, device=device)
b_gpu = torch.tensor([[3.0, 3.0, -3.0, -3.0]], dtype=torch.float32, device=device)

# One cuDNN handle is created here and reused for both graphs below.
handle = cudnn.create_handle()

print(f"a: {a_gpu.tolist()}")
print(f"b: {b_gpu.tolist()}")

In [ ]:
# Truncated modulo: y = mod(a, b). Result takes the sign of the dividend a (matches torch.fmod).
mod_graph = cudnn.pygraph(
    handle=handle,
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
)

a = mod_graph.tensor_like(a_gpu)
b = mod_graph.tensor_like(b_gpu)

y = mod_graph.mod(a, b, compute_data_type=cudnn.data_type.FLOAT, name="mod")
y.set_output(True).set_data_type(cudnn.data_type.FLOAT)

mod_graph.validate()
mod_graph.build_operation_graph()
mod_graph.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
mod_graph.check_support()
mod_graph.build_plans()

mod_gpu = torch.empty_like(a_gpu)
workspace = torch.empty(mod_graph.get_workspace_size(), device=device, dtype=torch.uint8)

mod_graph.execute({a: a_gpu, b: b_gpu, y: mod_gpu}, workspace, handle=handle)
torch.cuda.synchronize()

print(f"mod result: {mod_gpu.tolist()}")

In [ ]:
# torch.fmod is the truncated remainder (sign of the dividend) -- the reference for graph.mod.
ref_trunc = torch.fmod(a_gpu, b_gpu)

print(f"cuDNN mod:   {mod_gpu.tolist()}")
print(f"torch.fmod:  {ref_trunc.tolist()}")
assert torch.allclose(mod_gpu, ref_trunc), f"mod mismatch: {mod_gpu} vs {ref_trunc}"
print("PASSED: cuDNN mod matches torch.fmod (truncated remainder).")

In [ ]:
# Floored modulo: y = a - floor(a / b) * b. Result takes the sign of the divisor b (matches torch.remainder).
# On a cuDNN version / GPU that does not support floor_mod, the build raises
# cudnnGraphNotSupportedError, which we catch to skip gracefully.
try:
    floor_mod_graph = cudnn.pygraph(
        handle=handle,
        intermediate_data_type=cudnn.data_type.FLOAT,
        compute_data_type=cudnn.data_type.FLOAT,
    )

    a = floor_mod_graph.tensor_like(a_gpu)
    b = floor_mod_graph.tensor_like(b_gpu)

    y = floor_mod_graph.floor_mod(a, b, compute_data_type=cudnn.data_type.FLOAT, name="floor_mod")
    y.set_output(True).set_data_type(cudnn.data_type.FLOAT)

    floor_mod_graph.validate()
    floor_mod_graph.build_operation_graph()
    floor_mod_graph.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
    floor_mod_graph.check_support()
    floor_mod_graph.build_plans()

    floor_mod_gpu = torch.empty_like(a_gpu)
    workspace = torch.empty(floor_mod_graph.get_workspace_size(), device=device, dtype=torch.uint8)

    floor_mod_graph.execute({a: a_gpu, b: b_gpu, y: floor_mod_gpu}, workspace, handle=handle)
    torch.cuda.synchronize()

    print(f"floor_mod result: {floor_mod_gpu.tolist()}")
except cudnn.cudnnGraphNotSupportedError:
    floor_mod_gpu = None
    print("floor_mod not supported on this GPU / cuDNN version (requires cuDNN 9.26+) — skipping.")

In [ ]:
# torch.remainder is the floored remainder (sign of the divisor) -- the reference for graph.floor_mod.
if floor_mod_gpu is not None:
    ref_floor = torch.remainder(a_gpu, b_gpu)

    print(f"cuDNN floor_mod:   {floor_mod_gpu.tolist()}")
    print(f"torch.remainder:   {ref_floor.tolist()}")
    assert torch.allclose(floor_mod_gpu, ref_floor), f"floor_mod mismatch: {floor_mod_gpu} vs {ref_floor}"
    print("PASSED: cuDNN floor_mod matches torch.remainder (floored remainder).")
else:
    print("Skipped floor_mod reference check (op unsupported here).")

In [ ]:
# Side-by-side: mod and floor_mod agree on same-sign operands and differ on opposite-sign operands.
if floor_mod_gpu is not None:
    a_list = a_gpu.flatten().tolist()
    b_list = b_gpu.flatten().tolist()
    mod_list = mod_gpu.flatten().tolist()
    floor_list = floor_mod_gpu.flatten().tolist()

    print(f"{'a':>4} {'b':>4} {'mod (torch.fmod)':>18} {'floor_mod (torch.remainder)':>28}   note")
    print("-" * 66)
    for av, bv, mv, fv in zip(a_list, b_list, mod_list, floor_list):
        note = "differ" if abs(mv - fv) > 1e-6 else "agree"
        print(f"{av:4.0f} {bv:4.0f} {mv:18.0f} {fv:28.0f}   {note}")

    print("\nThe two flavors differ exactly on the opposite-sign rows (a=-7, b=3 and a=7, b=-3).")
else:
    print("floor_mod was not supported on this GPU / cuDNN version, so only the mod flavor ran above; skipping the side-by-side comparison.")

In [ ]:
cudnn.destroy_handle(handle)